# 5 · All in one - render, train detector and classifier, export

For a run **overnight** in **one** session:
1. render the full set (finished chunks are skipped),
2. train the detector (from scratch, or from `START_FROM`), measure, export,
3. build the classifier crops, train, compare, export,
4. then **end** the session (saves compute units).

**Before:** notebook 1 has run once (check confirmed). Then: *Runtime → Change runtime type → L4 GPU*,
*Runtime → Run all*. Keep the browser tab open and do not let the laptop sleep.

**Stopped?** Just *Run all* again: finished chunks, crops and training runs are skipped, started ones resume.
The log is shown below and also stored on Drive (`projects/…/all_in_one.log`).
The previous export files are backed up to `export_backup_<time>/` before they are overwritten.

## 1 · Settings
The trainer has filled in Drive folder and project. Same meaning as in notebooks 1-4.

In [ ]:
DRIVE_FOLDER = "__DRIVE__"        # folder in "My Drive"
PROJECT = "__PROJECT__"           # project folder name (under projects/)
NB_NAME, NB_VERSION = "5_all_in_one", 2   # version of this notebook (to detect updates)
BLENDER_SERIES = "4.5"            # Blender LTS series; the exact version is recorded the first time
IMAGES = 28000                    # renders in total
IMAGES_PER_CHUNK = 250            # rendered in one go, then saved to Drive
SAMPLES = 32                      # quality (more = nicer, but slower)
SEED = 1
SHARE_CLOSE = 0.35                # throws with dice lying close together
SHARE_LEAN_WALL = 0.2             # images with a die leaning against the wall (“?”)
SHARE_KEEP = 0.2                  # “keep and re-roll”: 2-4 dice pushed together at the wall, the rest thrown
SHARE_LEAN_DIE = 0.05             # images with a die leaning against another die (“?”)
# detector (as notebook 3)
EPOCHS = 300                # from scratch: passes over all images (upper limit)
PATIENCE = 100              # from scratch: stop when nothing improves for this many epochs
BATCH = 64
RUN = "run1"                # name of the training run (a new name = a new run, the old one stays)
START_FROM = ""             # continue from the best.pt of this run; "" (or missing) = from scratch
EPOCHS_CONTINUE = 80        # when continuing: upper limit …
PATIENCE_CONTINUE = 25      # … and patience
DET_DATA = "/content/det_data"   # local (fast), rebuilt in every session
# classifier (as notebook 4)
# name: (width, fast[, blur, color_cast]). "fast" = first stage with stride 2: half the compute.
# blur / color_cast (default 1.0) strengthen the degradation during training - for real camera images that are
# blurrier and have a stronger colour cast than the renders.
VARIANTS = {"s025": (0.25, True), "b025": (0.25, False), "b050": (0.5, False), "b100": (1.0, False),
            "s025f": (0.25, True, 1.0, 2.0),     # like s025, double colour cast
            "s025fu": (0.25, True, 1.5, 2.0)}    # like s025, double colour cast + 1.5× blur
EPOCHS_CLS = 60
TRAIN = ["s025f"]          # which variants are trained (compare all: list(VARIANTS))
CHOSEN = "s025f"           # exported variant; after the comparison (cell 8) change it if needed, then run cells 1 + 9
MODELS_FOLDER = "models"   # under classifier/; a new name keeps the older models for comparison
COMPARE_ALSO = []          # older models to measure along in cell 8, e.g. ["models_old/s025f"] (under classifier/)
CROPS_FOLDER = "crops"     # under classifier/; crops on Drive (built once, reused in every session)

END_SESSION = True          # at the end (also after an error) end the session - the results are on Drive

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
P = f"{DRIVE}/projects/{PROJECT}"
SCRIPTS = f"{DRIVE}/scripts"
SCENE = f"{P}/scene"
RENDERS = f"{P}/renders"
PREVIEW = f"{P}/renders_preview"
EXPORT = f"{P}/export"
DET = f"{P}/detector"
CLS = f"{P}/classifier"
ESP_DETECTION = "/content/esp-detection"
print("Project folder:", P)

## 2 · Check the GPU
Expected: **NVIDIA L4** (faster) or **Tesla T4**. If it says "no GPU": *Runtime → Change runtime type → L4 GPU*
(or T4), then start again from the top.

In [ ]:
import subprocess
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    gpu = ""
print("✔ GPU:", gpu) if gpu else print("✘ no GPU! Runtime → Change runtime type → L4 or T4 GPU, then run everything again.")

## 3 · Connect Google Drive and unpack the newest package
Colab asks whether the notebook may access your Drive → **Allow**.

Then the **newest Colab package** in `packages/` is unpacked (the ZIP file from the trainer - upload it to
*My Drive → __DRIVE__ → packages*, do **not** unpack it). Scripts, notebooks, `project.json`, photos, test images and the
camera setup are updated. **Renders and training results are not touched.**
A package is unpacked only once; a newer package replaces photos, test images and camera setup completely.

In [ ]:
import glob, json, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")
PACKAGES = f"{DRIVE}/packages"
os.makedirs(PACKAGES, exist_ok=True)
packages = sorted(glob.glob(f"{PACKAGES}/Colab_{PROJECT}_*.zip"))
state_file = f"{DRIVE}/package_state.json"
state = json.load(open(state_file)) if os.path.exists(state_file) else {}
if not packages:
    print(f"⚠ No package “Colab_{PROJECT}_….zip” in {PACKAGES} - create one in the trainer and upload the ZIP file there.")
elif state.get(PROJECT) == os.path.basename(packages[-1]):
    print("✔ newest package is unpacked already:", os.path.basename(packages[-1]))
else:
    new = packages[-1]
    print("unpacking:", os.path.basename(new))
    with zipfile.ZipFile(new) as z:
        names = [n for n in z.namelist() if not n.endswith("/")]
        prefix = names[0].split("/")[0]
        # mirror of the laptop: these folders are replaced completely (deleted photos disappear here too)
        for part in ("scripts", "notebooks", f"projects/{PROJECT}/photos", f"projects/{PROJECT}/test_images",
                     f"projects/{PROJECT}/camera_setup"):
            if any(n.startswith(f"{prefix}/{part}/") for n in names) and os.path.isdir(f"{DRIVE}/{part}"):
                shutil.rmtree(f"{DRIVE}/{part}")
        for n in names:
            target = os.path.join(DRIVE, n.split("/", 1)[1])
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(n) as src, open(target, "wb") as f:
                shutil.copyfileobj(src, f)
    state[PROJECT] = os.path.basename(new)
    json.dump(state, open(state_file, "w"), indent=1)
    print(f"✔ unpacked ({len(names)} files)")
    try:
        v = json.load(open(f"{DRIVE}/notebooks/version.json")).get(NB_NAME, 0)
        if v > NB_VERSION:
            print(f"⚠ The package contains a NEWER version of this notebook: please open notebooks/{NB_NAME}.ipynb in Drive again.")
    except (OSError, ValueError):
        pass
missing = [p for p in (f"{P}/project.json", f"{SCRIPTS}/render.py") if not os.path.exists(p)]
if missing:
    print("✘ not found:", *missing, sep="\n   ")
else:
    print("✔ project and scripts found")

## 4 · Install Blender (automatic, about 1-3 minutes)
Downloads Blender 4.5 LTS from **download.blender.org** and unpacks it on the Colab disk. The first time the download
is also stored in your Drive (`blender/`) - after that it is faster and it always stays **the same version**.
Colab forgets everything installed in every new session, so this cell runs every time.
At the end it shows whether Blender sees the GPU (expected: **OPTIX** or **CUDA**).

In [ ]:
import json, os, re, shutil, subprocess, urllib.request
CACHE = f"{DRIVE}/blender"
os.makedirs(CACHE, exist_ok=True)
vfile = f"{CACHE}/version.json"
if os.path.exists(vfile):
    version = json.load(open(vfile))["version"]
else:
    html = urllib.request.urlopen(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/").read().decode()
    found = set(re.findall(rf"blender-({re.escape(BLENDER_SERIES)}\.\d+)-linux-x64\.tar\.xz", html))
    version = sorted(found, key=lambda v: int(v.split(".")[-1]))[-1]
    json.dump({"version": version}, open(vfile, "w"))
archive = f"blender-{version}-linux-x64.tar.xz"
if not os.path.exists(f"{CACHE}/{archive}"):
    print("downloading", archive, "…")
    urllib.request.urlretrieve(f"https://download.blender.org/release/Blender{BLENDER_SERIES}/{archive}", f"/content/{archive}")
    shutil.copy(f"/content/{archive}", f"{CACHE}/{archive}")
BLENDER_DIR = f"/content/blender-{version}-linux-x64"
if not os.path.isdir(BLENDER_DIR):
    source = f"/content/{archive}" if os.path.exists(f"/content/{archive}") else f"{CACHE}/{archive}"
    subprocess.run(["tar", "-xf", source, "-C", "/content"], check=True)
subprocess.run("apt-get -qq update > /dev/null; apt-get -qq install -y libxi6 libxkbcommon0 libsm6 libxrender1 "
               "libxxf86vm1 libxfixes3 libgl1 libegl1 > /dev/null", shell=True)
BLENDER = f"{BLENDER_DIR}/blender"
print(subprocess.run([BLENDER, "-b", "--version"], capture_output=True, text=True).stdout.splitlines()[0])
test = ("import bpy; p=bpy.context.preferences.addons['cycles'].preferences\n"
        "for t in ('OPTIX','CUDA'):\n"
        "    try:\n        p.compute_device_type=t; p.get_devices()\n"
        "        g=[d.name for d in p.devices if d.type==t]\n"
        "        if g: print('GPU RENDERING:', t, g); break\n"
        "    except TypeError: pass\n"
        "else: print('GPU RENDERING: none - rendering on the CPU (very slow)')")
out = subprocess.run([BLENDER, "-b", "--python-expr", test], capture_output=True, text=True).stdout
print([z for z in out.splitlines() if z.startswith("GPU RENDERING")] or out[-500:])

## 5 · Install the training programs (about 1-2 minutes, every session)
Fixed versions so that the result stays reproducible: **ultralytics 8.4.189**, **albumentations 2.0.8**,
**onnx 1.23.2**, **onnxsim 0.8.1** and Espressif's **esp-detection** (fixed commit), which describes the detector
*ESPDet-Pico*. PyTorch comes with Colab.

In [ ]:
import os, subprocess, sys, json
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.189", "albumentations==2.0.8",
                "onnx==1.23.2", "onnxsim==0.8.1"], check=True)
if not os.path.isdir(ESP_DETECTION):
    subprocess.run(["git", "clone", "-q", "https://github.com/espressif/esp-detection.git", ESP_DETECTION], check=True)
subprocess.run(["git", "-C", ESP_DETECTION, "checkout", "-q", "28de5902a357d99dd9a5db0f52905255641c5a78"], check=True)
sys.path.insert(0, SCRIPTS)
import torch, ultralytics, ml_tools
d = ml_tools.load_project(P)
IMGSZ, CROP = ml_tools.sizes(d)
print(f"✔ torch {torch.__version__} (CUDA: {torch.cuda.is_available()}), ultralytics {ultralytics.__version__}")
print(f"✔ classes: {', '.join(ml_tools.classes(d))} · detector input {IMGSZ} px · crop {CROP} px")

## 6 · All steps
All steps one after the other. For every step a heading with the time, shortened output and the duration. If a step
fails, the following steps stop (so that nothing runs on half the data) - the message is below and in the log.
Afterwards the session is ended (`END_SESSION`).

In [ ]:
import hashlib, json, os

def atlas_hash():
    return hashlib.sha1(open(f"{SCENE}/atlas.png", "rb").read() + open(f"{SCENE}/scene.json", "rb").read()).hexdigest()

def check_confirmed():
    f = f"{SCENE}/check_ok.json"
    return (os.path.exists(f) and os.path.exists(f"{SCENE}/atlas.png")
            and json.load(open(f)).get("hash") == atlas_hash())

import datetime, math, re, shutil, subprocess, time
LOG = f"{P}/all_in_one.log"
CLS_DATA = f"{CLS}/{CROPS_FOLDER}"
MODELS = f"{CLS}/{MODELS_FOLDER}"
start_total = time.time()

def log(text):
    line = f"{datetime.datetime.now():%H:%M:%S} {text}"
    print(line, flush=True)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def step(title, cmd, filt=lambda z: True):
    log(f"=== {title}")
    t0 = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        z = re.sub(r"\x1b\[[0-9;]*[A-Za-z]", "", line.rstrip("\n").split("\r")[-1])
        tail = (tail + [z])[-40:]
        if z.strip() and filt(z):
            print(z, flush=True)
    proc.wait()
    log(f"    {title}: {(time.time() - t0) / 60:.1f} min · return code {proc.returncode}")
    if proc.returncode:
        with open(LOG, "a", encoding="utf-8") as f:
            f.write("\n".join(tail) + "\n")
        raise RuntimeError(f"step “{title}” failed (last lines above and in {LOG})")

def backup_export():
    if os.path.isdir(EXPORT) and os.listdir(EXPORT):
        target = f"{P}/export_backup_{datetime.datetime.now():%Y%m%d_%H%M%S}"
        shutil.copytree(EXPORT, target)
        log(f"    previous export backed up in {target}")

try:
    log(f"All in one {PROJECT} started · GPU {gpu or '-'}")
    # ---------------------------------------------------------------- 1. render
    if not (os.path.exists(f"{SCENE}/scene.json") and check_confirmed()):
        raise RuntimeError("Texture atlas not confirmed - first notebook 1, cells 5-7.")
    chunks = math.ceil(IMAGES / IMAGES_PER_CHUNK)
    step(f"Render ({IMAGES} images, chunks 0-{chunks - 1})",
         [BLENDER, "-b", "-P", f"{SCRIPTS}/render.py", "--", "--scene", SCENE, "--output", RENDERS,
          "--chunk-from", "0", "--chunk-to", str(chunks - 1), "--images", str(IMAGES_PER_CHUNK),
          "--seed", str(SEED), "--samples", str(SAMPLES), "--share-close", str(SHARE_CLOSE), "--share-lean-wall", str(SHARE_LEAN_WALL), "--share-keep", str(SHARE_KEEP), "--share-lean-die", str(SHARE_LEAN_DIE)],
         lambda z: "✔" in z or "⚠" in z or "skipped" in z or "ERROR" in z or "Error" in z or "render device" in z or "image 1/" in z)
    step("Render statistics", ["python", f"{SCRIPTS}/render_stats.py", "--renders", RENDERS, "--scene", SCENE])
    # ---------------------------------------------------------------- 2. detector
    step("Detector: build the dataset", ["python", f"{SCRIPTS}/det_dataset.py", "--project", P, "--renders", RENDERS,
                                         "--output", DET_DATA], lambda z: "✔" in z or "⚠" in z)
    start = f"{DET}/runs/{START_FROM}/weights/best.pt" if START_FROM else ""
    run = f"{DET}/runs/{RUN}"
    base = ["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
            "--name", RUN, "--imgsz", str(IMGSZ), "--esp-detection", ESP_DETECTION]
    if os.path.exists(f"{run}/result.json"):
        log(f"    detector run {RUN} is finished already - training skipped")
    else:
        cmd = base + ["--batch", str(BATCH)]
        if os.path.exists(f"{run}/weights/last.pt"):
            cmd.append("--resume")
        elif start and os.path.exists(start):
            cmd += ["--start-from", start, "--epochs", str(EPOCHS_CONTINUE), "--patience", str(PATIENCE_CONTINUE)]
        else:
            if start:
                log(f"    {start} is missing - the detector is trained from scratch")
            cmd += ["--epochs", str(EPOCHS), "--patience", str(PATIENCE)]
        step(f"Detector: train ({RUN})", cmd, lambda z: z.startswith("  epoch ") or "final validation" in z or "✔" in z or "⏱" in z or "rror" in z
             or "continuing" in z or "Resuming" in z)
    step(f"Detector: measure ({RUN})", base + ["--test-only"], lambda z: "✔" in z)
    if start and START_FROM != RUN and os.path.exists(start):
        step(f"Detector: measure ({START_FROM}, before)",
             ["python", f"{SCRIPTS}/det_train.py", "--data", f"{DET_DATA}/data.yaml", "--output", f"{DET}/runs",
              "--name", START_FROM, "--imgsz", str(IMGSZ), "--esp-detection", ESP_DETECTION, "--test-only",
              "--result", "result_compare.json"], lambda z: "✔" in z)
    backup_export()
    step("Detector: export", ["python", f"{SCRIPTS}/det_export.py", "--weights", f"{run}/weights/best.pt",
                              "--project", P, "--renders", RENDERS, "--esp-detection", ESP_DETECTION,
                              "--imgsz", str(IMGSZ), "--output", EXPORT], lambda z: "✔" in z or "rror" in z)
    # ---------------------------------------------------------------- 3. classifier
    step("Classifier: crops (only new ones)", ["python", f"{SCRIPTS}/cls_crops.py", "--project", P, "--renders", RENDERS,
                                                "--output", CLS_DATA, "--size", str(CROP)],
         lambda z: "✔" in z or "⚠" in z or "crops from" in z or "rebuilt" in z)
    for name in TRAIN:
        width, fast, *extra = VARIANTS[name]
        blur, color_cast = (extra + [1.0, 1.0])[:2]
        if os.path.exists(f"{MODELS}/{name}.done.json"):
            log(f"    {name} is done already - skipped")
            continue
        cmd = ["python", f"{SCRIPTS}/cls_train.py", "--data", CLS_DATA, "--output", MODELS, "--name", name,
               "--width", str(width), "--epochs", str(EPOCHS_CLS)] + (["--fast"] if fast else []) + \
              (["--blur", str(blur)] if blur != 1.0 else []) + (["--color-cast", str(color_cast)] if color_cast != 1.0 else [])
        if width >= 1.0:
            cmd += ["--lr", "0.001"]
        if os.path.exists(f"{MODELS}/{name}.ckpt"):
            cmd.append("--resume")
        step(f"Classifier: train ({name})", cmd, lambda z: "poch" in z or "✔" in z or "rror" in z)
    done = [n for n in VARIANTS if os.path.exists(f"{MODELS}/{n}.done.json")]
    also = [f"{CLS}/{a}.pth" for a in COMPARE_ALSO if os.path.exists(f"{CLS}/{a}.pth")]
    step("Classifier: compare", ["python", f"{SCRIPTS}/cls_train.py", "--data", CLS_DATA, "--output", MODELS,
                                 "--compare", *done, *also], lambda z: False)
    step(f"Classifier: export ({CHOSEN})", ["python", f"{SCRIPTS}/cls_export.py", "--model", f"{MODELS}/{CHOSEN}.pth",
                                            "--data", CLS_DATA, "--project", P, "--output", EXPORT],
         lambda z: "✔" in z or "rror" in z)
    # ---------------------------------------------------------------- summary
    log("=== Result")
    for name, fname in ((RUN, "result.json"), (START_FROM, "result_compare.json")):
        f = f"{DET}/runs/{name}/{fname}"
        if name and os.path.exists(f) and (name == RUN or START_FROM != RUN):
            e = json.load(open(f))
            for k in ("synthetic", "real"):
                if k in e:
                    log(f"    detector {name:8s} {k:10s}: mAP50 {e[k]['mAP50']:.3f} · recall {e[k]['recall']:.3f} · "
                        f"precision {e[k]['precision']:.3f}")
    comparison = open(f"{MODELS}/compare.md", encoding="utf-8").read()
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(comparison + "\n")
    try:
        from IPython.display import Markdown, display
        display(Markdown(comparison))
    except ImportError:
        print(comparison)
    for f in sorted(os.listdir(EXPORT)):
        log(f"    export/{f:24s} {os.path.getsize(os.path.join(EXPORT, f)) / 1024:8.1f} KB")
    log(f"✔ All in one done after {(time.time() - start_total) / 3600:.1f} h - continue on the laptop (trainer, “Training & device”)")
except Exception as e:
    log(f"✘ stopped: {e}")
finally:
    if END_SESSION:
        log("Ending the session (END_SESSION = True).")
        from google.colab import runtime
        runtime.unassign()